In [42]:
# external lib imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neural_network import MLPClassifier


SEED_VALUE = 36
np.random.seed(SEED_VALUE)


In [43]:
# internal lib import
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from core import NeuralNetwork


In [44]:
# Load the dataset
df = pd.read_csv("../data/datasetml_2026.csv")

In [45]:
# Separate Features (X) and Target (y)
X = df.drop(columns=['placement_status'])
y = df['placement_status'].values
y = np.where(y == "Placed", 1, 0) 

# Preprocess Categorical and Numerical Columns
categorical_cols = ['college_tier', 'country', 'university_ranking_band', 'specialization', 'industry']
numerical_cols = ['cgpa', 'backlogs', 'internship_count', 'aptitude_score', 'communication_score', 'internship_quality_score']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=SEED_VALUE)

# Reshape y for the Neural Network to (batch_size, 1)
y_train = y_train.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


X_train shape: (8000, 23)
y_train shape: (8000, 1)


# TEST VERBOSE

In [46]:
input_dim = X_train.shape[1]

model_test = NeuralNetwork(
    layer_sizes=[input_dim, 16, 1],
    activations=['relu', 'sigmoid'],
    loss='mse',
    seed=SEED_VALUE
)

print("Starting training to test verbose=1 output...")
history = model_test.fit(
    X_train, y_train,
    val_data=(X_val, y_val),
    batch_size=32,
    learning_rate=0.01,
    optimizer='sgd',
    epochs=200,
    verbose=1
)

Starting training to test verbose=1 output...
Epoch 1/200 [>.............................] - train_loss: 0.233619 - val_loss: 0.233006

Epoch 200/200 [==============================>] - train_loss: 0.158283 - val_loss: 0.168447


# TEST SAVE & LOAD


In [47]:
import os

save_path = "test_model_save.json"

# 1. Save the model
print(f"Saving the model to {save_path}...")
model_test.save(save_path)

# 2. Load the model back
print("Loading the model...")
loaded_model = NeuralNetwork.load(save_path)

# 3. Verify they are identical by comparing predictions on validation data
preds_original = model_test.predict(X_val)
preds_loaded = loaded_model.predict(X_val)

matches = np.allclose(preds_original, preds_loaded)
print(f"Predictions match exactly after loading: {matches}")

# 4. Clean up the test file
if os.path.exists(save_path):
    os.remove(save_path)
    print("Cleaned up the temporary model file.")

Saving the model to test_model_save.json...
Loading the model...
Predictions match exactly after loading: True
Cleaned up the temporary model file.
